# Ejercicio 4

a. Implemente una red con aprendizaje Backpropagation que aprenda la siguiente función:
$$ f(x,y,z) = \sin(x) + \cos(y) + z $$
donde: $x$ e $y \in [0, 2 \pi ]$ y $z \in [− 1,1]$. Para ello construya un conjunto de datos de entrenamiento y un conjunto de evaluación. Muestre la evolución del error de entrenamiento y de evaluación en función de las épocas de entrenamiento.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(110007)

In [ ]:
class MLP:
    # layers incluye la entrada, por ejemplo (4, 8, 2) son 4 entradas,
    # una capa oculta con 8 neuronas, y 2 neuronas de salida
    def __init__(self, layers, linear_output=False):
        self.weights = []
        self.biases = []
        self.n_layers = len(layers) - 1
        self.linear_output = linear_output
        for i in range(self.n_layers):
            n_in = layers[i]
            n_out = layers[i + 1]
            self.weights.append(np.random.normal(0, 1, size=(n_in, n_out)))
            self.biases.append(np.random.normal(0, 1, size=(1, n_out)))
        self.losses = []
        self.losses_test = []

    def predict(self, X):
        V = self.forward(X)
        return V[-1]
    
    def forward(self, X):
        V = [X]
        for m in range(self.n_layers):
            h = V[-1] @ self.weights[m] + self.biases[m]
            if m == self.n_layers - 1 and self.linear_output:
                Vm = h
            else:
                Vm = self.activation(h)
            V.append(Vm)
        return V

    def backward(self, X, y, V, lr):
        m = X.shape[0]
        
        Vout = V[-1]
        y_ = y.reshape(Vout.shape)
        error = Vout - y_
        
        if self.linear_output:
            delta = error
        else:
            delta = error * self.activation_dif(Vout)
        
        for i in reversed(range(self.n_layers)):
            Vprev = V[i]
            
            dW = (Vprev.T @ delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m
            
            self.weights[i] -= lr * dW
            self.biases[i] -= lr * db
            
            if i > 0:
                delta = (delta @ self.weights[i].T) * self.activation_dif(Vprev)

    def train(self, X, y, X_test=None, y_test=None, epochs=1000, lr=0.1):
        for e in range(epochs):
            V = self.forward(X)
            self.backward(X, y, V, lr)
            train_err = self.error(X, y)
            self.losses.append(train_err)

            if X_test is not None and y_test is not None:
                test_err = self.error(X_test, y_test)
                self.losses_test.append(test_err)
            else:
                test_err = 0.0

            if e % 100 == 0:
                print(f"Epoch {e:5d}: train error: {train_err:4.3g}, test error: {test_err:4.3g}")

    def minibatch_train(self, X_train, y_train, batch_size, epochs=100, lr=0.1, X_test=None, y_test=None):
        n_samples = X_train.shape[0]
        
        for e in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]
            
            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i : i + batch_size]
                y_batch = y_shuffled[i : i + batch_size]
                
                V = self.forward(X_batch)
                self.backward(X_batch, y_batch, V, lr)
                
            train_err = self.error(X_train, y_train)
            self.losses.append(train_err)
            
            if X_test is not None and y_test is not None:
                test_err = self.error(X_test, y_test)
                self.losses_test.append(test_err)
            else:
                test_err = 0.0
            
            if e % 100 == 0:
                print(f"Epoch {e:5d}: train error: {train_err:4.3g}, test error: {test_err:4.3g}")

    def error(self, X, y):
        y_pred = self.predict(X).reshape(y.shape)
        return np.mean((y_pred - y) ** 2)

    @staticmethod
    def activation(h):
        return 1 / (1 + np.exp(-h))

    @staticmethod
    def activation_dif(V):
        return V * (1 - V)

In [ ]:
num_samples = 1000
X = np.random.uniform([0, 0, -1], [2*np.pi, 2*np.pi, 1], size=(num_samples, 3))
y = np.sin(X[:,0]) + np.cos(X[:,1]) + X[:,2]

train_fraction = 0.9
Np_train = int(train_fraction*num_samples)
Np_test = num_samples - Np_train

X_train = X[:Np_train,:]
y_train = y[:Np_train]

X_test = X[Np_train:,:]
y_test = y[Np_train:]

In [ ]:
model = MLP(layers=[3, 30, 1], linear_output=True)
model.train(X_train, y_train, X_test=X_test, y_test=y_test, lr=0.05, epochs=10000)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses, label='Error de entrenamiento')
plt.plot(model.losses_test, label='Error de evaluación')
plt.grid()
plt.legend()
plt.xlabel('Época')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('../informe/img/ej4/loss.svg')
plt.show()

In [ ]:
y_pred = model.predict(X_test)

plt.figure(figsize=(4,4))
plt.scatter(y_test, y_pred)
plt.plot([-3, 3], [-3, 3], c='tab:orange')
plt.xlim([-3, 3])
plt.ylim([-3, 3])
plt.grid()
plt.xlabel('Salida deseada')
plt.ylabel('Salida esperada')
plt.tight_layout()
plt.savefig('../informe/img/ej4/prediction.svg')
plt.show()

In [ ]:
plt.figure(figsize=(4,4))
x = np.linspace(0, np.pi*2, 100)
X = np.c_[x, np.ones(100)*np.pi, np.ones(100)/2]
y = model.predict(X)
plt.plot(x, y, label='Predicción')
plt.plot(x, np.sin(x)+np.cos(np.pi)+0.5, label='Esperada')
plt.grid()
plt.legend()
plt.xlabel('Entrada $x$')
plt.ylabel('Salida de la red')
plt.tight_layout()
plt.savefig('../informe/img/ej4/output.svg')
plt.show()

b) Estudie la evolución de los errores durante el entrenamiento de una red con una capa oculta de 30 neuronas cuando el conjunto de entrenamiento contiene 40 muestras. ¿Que ocurre si el minibatch tiene tamaño 40? ¿Y si tiene tamaño 1?

In [ ]:
np.random.seed(110007)

In [ ]:
num_samples = 60
X = np.random.uniform([0, 0, -1], [2*np.pi, 2*np.pi, 1], size=(num_samples, 3))
y = np.sin(X[:,0]) + np.cos(X[:,1]) + X[:,2]

Np_train = 40
Np_test = num_samples - Np_train

X_train = X[:Np_train,:]
y_train = y[:Np_train]

X_test = X[Np_train:,:]
y_test = y[Np_train:]

In [ ]:
model = MLP(layers=[3, 30, 1], linear_output=True)
model.minibatch_train(X_train, y_train, 40, X_test=X_test, y_test=y_test, lr=0.05, epochs=10000)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses, label='Error de entrenamiento')
plt.plot(model.losses_test, label='Error de evaluación')
plt.grid()
plt.legend()
plt.xlabel('Época')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('../informe/img/ej4/loss_40.svg')
plt.show()

In [ ]:
model = MLP(layers=[3, 30, 1], linear_output=True)
model.minibatch_train(X_train, y_train, 1, X_test=X_test, y_test=y_test, lr=0.01, epochs=10000)

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(model.losses, label='Error de entrenamiento')
plt.plot(model.losses_test, label='Error de evaluación')
plt.grid()
plt.legend()
plt.xlabel('Época')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('../informe/img/ej4/loss_1.svg')
plt.show()